In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/sobhanmoosavi/us-accidents/US_Accidents_March23.csv


In [2]:
df= pd.read_csv("/kaggle/input/datasets/sobhanmoosavi/us-accidents/US_Accidents_March23.csv")

In [47]:
df.head()
df['State'].value_counts()

State
CA    1741433
FL     880192
TX     582837
SC     382557
NY     347960
NC     338199
VA     303301
PA     296620
MN     192084
OR     179660
AZ     170609
GA     169234
IL     168958
TN     167388
MI     162191
LA     149701
NJ     140719
MD     140417
OH     118115
WA     108221
AL     101044
UT      97079
CO      90885
OK      83647
MO      77323
CT      71005
IN      67224
MA      61996
WI      34688
KY      32254
NE      28870
MT      28496
IA      26307
AR      22780
NV      21665
KS      20992
DC      18630
RI      16971
MS      15181
DE      14097
WV      13793
ID      11376
NM      10325
NH      10213
WY       3757
ND       3487
ME       2698
VT        926
SD        289
Name: count, dtype: int64

In [113]:
df_fl= df[df['State'] == 'FL']

In [114]:
df_fl.shape

(880192, 46)

In [115]:
columns= ['Severity', 'Start_Time', 'Start_Lat','Start_Lng',  'Temperature(F)', 'Wind_Chill(F)', 'Humidity(%)', 'Pressure(in)', 'Visibility(mi)', 'Wind_Speed(mph)','Precipitation(in)', 'Weather_Condition',  'Crossing',  'Junction', 'Railway',  'Stop', 'Traffic_Signal', 'Sunrise_Sunset']

In [117]:

df_fl=df_fl[columns]

In [118]:
df_fl.columns

Index(['Severity', 'Start_Time', 'Start_Lat', 'Start_Lng', 'Temperature(F)',
       'Wind_Chill(F)', 'Humidity(%)', 'Pressure(in)', 'Visibility(mi)',
       'Wind_Speed(mph)', 'Precipitation(in)', 'Weather_Condition', 'Crossing',
       'Junction', 'Railway', 'Stop', 'Traffic_Signal', 'Sunrise_Sunset'],
      dtype='object')

In [119]:
df_fl.notnull().sum()

Severity             880192
Start_Time           880192
Start_Lat            880192
Start_Lng            880192
Temperature(F)       866364
Wind_Chill(F)        690834
Humidity(%)          864709
Pressure(in)         872646
Visibility(mi)       868785
Wind_Speed(mph)      846633
Precipitation(in)    712523
Weather_Condition    870292
Crossing             880192
Junction             880192
Railway              880192
Stop                 880192
Traffic_Signal       880192
Sunrise_Sunset       875602
dtype: int64

In [120]:
df_fl.drop(columns=['Wind_Chill(F)', 'Precipitation(in)'], inplace=True)

In [121]:
df_Fl.fillna({
    'Temperature(F)': df_fl['Temperature(F)'].mean(),
    'Humidity(%)': df_fl['Humidity(%)'].mean(),
    'Pressure(in)': df_fl['Pressure(in)'].mean(),
    'Visibility(mi)': df_fl['Visibility(mi)'].mean(),
    'Wind_Speed(mph)': df_fl['Wind_Speed(mph)'].mean()
},inplace=True)

In [122]:
df_Fl.fillna({
    'Weather_Condition': df_fl['Weather_Condition'].mode()[0],
    'Sunrise_Sunset': df_fl['Sunrise_Sunset'].mode()[0]
},inplace=True)

In [124]:
df_fl['Start_Time'] = pd.to_datetime(df_fl['Start_Time'], errors='coerce')
df_fl['hour']= df_fl['Start_Time'].dt.hour
df_fl['day']= df_fl['Start_Time'].dt.dayofweek
df_fl['month']= df_fl['Start_Time'].dt.month

In [125]:
df_fl.sample(12)
df_fl.notnull().sum()
df_fl.fillna({
    'hour': df_fl['hour'].mode()[0],
    'day': df_fl['day'].mode()[0],
    'month': df_fl['month'].mode()[0]
}, inplace=True)

In [126]:
df_fl.notnull().sum()
df_fl.drop(columns= ['Start_Time'], inplace=True)

In [127]:

df_fl.notnull().sum()

Severity             880192
Start_Lat            880192
Start_Lng            880192
Temperature(F)       866364
Humidity(%)          864709
Pressure(in)         872646
Visibility(mi)       868785
Wind_Speed(mph)      846633
Weather_Condition    870292
Crossing             880192
Junction             880192
Railway              880192
Stop                 880192
Traffic_Signal       880192
Sunrise_Sunset       875602
hour                 880192
day                  880192
month                880192
dtype: int64

In [128]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer

ohe= OneHotEncoder(handle_unknown='ignore',sparse_output=False)

In [129]:
cat_cols= ['Weather_Condition',  'Crossing',  'Junction', 'Railway',  'Stop', 'Traffic_Signal', 'Sunrise_Sunset']
preprocessor= ColumnTransformer(transformers=[
    ('cat', ohe, cat_cols)
], remainder= 'passthrough')   
 

In [130]:
x= df_fl.drop(columns=['Severity'])
y= df_fl['Severity']


In [131]:
df_fl['Severity'].value_counts()

Severity
2    755895
3    104065
4     13149
1      7083
Name: count, dtype: int64

In [132]:
y1=df_fl['Severity']-1

In [133]:
y1.value_counts()

Severity
1    755895
2    104065
3     13149
0      7083
Name: count, dtype: int64

In [134]:
y1 = (df_fl['Severity'] >= 3).astype(int)

In [135]:
y1.value_counts()

Severity
0    762978
1    117214
Name: count, dtype: int64

In [142]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix

x_train, x_test, y_train, y_test = train_test_split(
    x, y1, test_size=0.2, random_state=42
)


# pipeline
model2 = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(
        objective='binary:logistic',
        n_estimators=300,
        max_depth=8,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        min_child_weight=3,
        gamma=0.1,
        reg_lambda=1.0,
        n_jobs=-1,
        random_state=42,
        scale_pos_weight = len(y_train[y_train==0]) / len(y_train[y_train==1])
    ))
])

# train
model2.fit(x_train, y_train)

# predict (0 or 1)
# y_pred = model2.predict(x_test)
y_prob = model2.predict_proba(x_test)[:, 1]
y_pred = (y_prob > 0.5).astype(int)

# evaluate
print("Accuracy:", model2.score(x_test, y_test))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.8257772425428456

Classification Report:
               precision    recall  f1-score   support

           0       0.97      0.82      0.89    152519
           1       0.43      0.86      0.57     23520

    accuracy                           0.83    176039
   macro avg       0.70      0.84      0.73    176039
weighted avg       0.90      0.83      0.85    176039


Confusion Matrix:
 [[125057  27462]
 [  3208  20312]]


In [143]:
import joblib
joblib.dump(model2,"florida_model1.pkl")

['florida_model1.pkl']

In [137]:
import numpy as np

df_fl = df_fl.copy()

# remove impossible coordinates
df_fl = df_fl[
    (df_fl['Start_Lat'].between(-90, 90)) &
    (df_fl['Start_Lng'].between(-180, 180))
]

# remove impossible weather values
df_fl = df_fl[df_fl['Temperature(F)'] > -100]   # sanity check
df_fl = df_fl[df_fl['Wind_Speed(mph)'] >= 0]
df_fl = df_fl[df_fl['Pressure(in)'] > 0]
df_fl = df_fl[df_fl['Humidity(%)'].between(0, 100)]
df_fl = df_fl[df_fl['Visibility(mi)'] >= 0]

In [138]:
def iqr_filter(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    return df[(df[col] >= lower) & (df[col] <= upper)]

numeric_cols = [
    'Temperature(F)',
    'Wind_Speed(mph)',
    'Pressure(in)',
    'Visibility(mi)',
    'Humidity(%)'
]

for col in numeric_cols:
    df_fl = iqr_filter(df_fl, col)

In [140]:
lat_min, lat_max = df_fl['Start_Lat'].quantile([0.01, 0.99])
lng_min, lng_max = df_fl['Start_Lng'].quantile([0.01, 0.99])

df_fl = df_fl[
    (df_fl['Start_Lat'].between(lat_min, lat_max)) &
    (df_fl['Start_Lng'].between(lng_min, lng_max))
]

In [141]:
df_Fl.shape

(710200, 18)